<a href="https://colab.research.google.com/github/sanjay0423/AgenticAI/blob/main/Sanjay_LangChain_end_to_end_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangChain End-to-End: Prompt → Chain → RAG → Agent → UI

This single notebook merges the **basics** and **agent** demos into one coherent flow:

1. **Hello LLM** (baseline)  
2. **Prompting + LCEL (Langchain Expression Language "|") + Output Parser**  --> Simple LLM Chain
3. **RAG (build once, re-use)** with sources  --> Retrieval Chain
4. **Agent + Tools** (retriever tool + optional web search)  
5. **Tiny UI** (Gradio)  
6. **Custom Tool (@tool) example**  


It will provide an end-to-end guide to building AI applications with LangChain and related tools. It covers:

*   **Setup**: Installing dependencies and configuring API keys.
*   **LLM Fundamentals**: Basic LLM calls, prompting techniques, and output parsing with LangChain Expression Language (LCEL).
*   **Retrieval-Augmented Generation (RAG)**: Building a RAG system from web documents using vector stores (FAISS) and embeddings.
*   **Agents and Tools**: Creating an intelligent agent capable of using custom tools (e.g., arithmetic), a knowledge base retriever, and optional web search.
*   **User Interface**: Integrating the agent into a simple Gradio-based chat interface.

It demonstrates how to progressively build a more sophisticated application, from a simple LLM call to a dynamic agent capable of complex interactions.

## 0) Install & Configure (run once)

- Installs pinned versions to reduce API drift.  
- Prompts for your keys as needed.  
- **Optional**: If you have a LangSmith key, tracing will be enabled automatically.

> If you re-run the notebook later, you can skip re-installation if your environment already has these packages.

In [ ]:
import os
import warnings

# 1. Tell Python to ignore this specific warning globally
os.environ['PYTHONWARNINGS'] = "ignore::DeprecationWarning:jupyter_client.session"

# 2. Silencing it in the current running process
warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client.session")

print("✅ Warning suppressed. You can proceed without the clutter.")

✅ Warning suppressed. You can proceed without the clutter.


In [ ]:
# @title Re-Install Jupyter Core Dependencies If Needed

# Upgrade to the version that supports Python 3.12 properly
%pip install --upgrade jupyter_client jupyter_core

# Force restart to load the new version into memory
import os
os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: jupyter_client
    Found existing installation: jupyter_client 7.4.9
    Uninstalling jupyter_client-7.4.9:
      Successfully uninstalled jupyter_client-7.4.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-kernel-gateway 2.5.2 requires jupyter-client<8.0,>=5.2.0, but you have jupyter-client 8.8.0 which is incompatible.
notebook 6.5.7 requires jupyter-client<8,>=5.3.4, but you have jupyter-client 8.8.0 which is incompatible.


In [ ]:
import jupyter_client
import pkg_resources
import requests

# 1. Check your current version
current = jupyter_client.__version__

# 2. Fetch the latest version from PyPI
response = requests.get("https://pypi.org/pypi/jupyter-client/json")
latest = response.json()["info"]["version"]

print(f"Current version in Colab: {current}")
print(f"Latest version on PyPI:   {latest}")

if pkg_resources.parse_version(current) < pkg_resources.parse_version(latest):
    print("\n🚀 An update is available! Use the %pip command to upgrade.")
else:
    print("\n✅ You are on the latest version.")

Current version in Colab: 8.8.0
Latest version on PyPI:   8.8.0

✅ You are on the latest version.


In [ ]:
# @title Install Dependencies
%%capture
%pip install -qqqq \
    beautifulsoup4 \
    faiss-cpu \
    "gradio>=4.0" \
    "langchain>=0.3.16" \
    "langchain-community>=0.3.16" \
    "langchain-openai>=0.2.14" \
    "langchain-text-splitters>=0.3.4" \
    "langgraph>=0.2.59" \
    langchainhub \
    lxml \
    "requests>=2.32.5" \
    tavily-python \
    python-dotenv \
    bcrypt

In [ ]:
%pip install langchain-openai>=0.2.14

In [ ]:
%pip show langchain-openai bcrypt tavily-python

Name: langchain-openai
Version: 1.1.10
Summary: An integration package connecting OpenAI and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/openai
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, openai, tiktoken
Required-by: 
---
Name: bcrypt
Version: 5.0.0
Summary: Modern password hashing for your software and your servers
Home-page: https://github.com/pyca/bcrypt/
Author: 
Author-email: The Python Cryptographic Authority developers <cryptography-dev@python.org>
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: 
Required-by: 
---
Name: tavily-python
Version: 0.7.21
Summary: Python wrapper for the Tavily API
Home-page: https://github.com/tavily-ai/tavily-python
Author: Tavily AI
Author-email: support@tavily.com
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: httpx, requests, tiktoken
Required-by: 


In [ ]:
# @title Verify Installations
import importlib
import warnings
# warnings.filterwarnings("ignore", category=DeprecationWarning)
# Suppress specific DeprecationWarnings
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    message=r".*datetime.*")

# Suppress specific DeprecationWarnings
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    message=r".*swigvarlink.*")

def _ver(name):
    try:
        m = importlib.import_module(name)
        return getattr(m, "__version__", "n/a")
    except Exception as e:
        return f"not installed ({e})"
print("langchain           :", _ver("langchain"))
print("langgraph           :", _ver("langgraph"))
print("langchain-core      :", _ver("langchain_core"))
print("langchain-community :", _ver("langchain_community"))
print("langchain-openai    :", _ver("langchain_openai"))
print("langchainhub        :", _ver("langchainhub"))
print("langchain-text-splitters:", _ver("langchain_text_splitters"))
print("faiss-cpu           :", _ver("faiss"))
print("tavily-python       :", _ver("tavily"))
print("bcrypt              :", _ver("bcrypt"))

langchain           : 1.2.10
langgraph           : n/a
langchain-core      : 1.2.13
langchain-community : 0.4.1
langchain-openai    : n/a
langchainhub        : n/a
langchain-text-splitters: n/a
faiss-cpu           : 1.13.2
tavily-python       : n/a
bcrypt              : 5.0.0


### Loading Environment Variables

There are options to load environment variables:
- `userdata` if running code on colab **(Recommended)**
- `getpass` module if running an interactive script
- `.env` file if running in production

In [ ]:
import os
from google.colab import userdata

# Access the variables
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] =  userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = userdata.get("LANGSMITH_PROJECT")
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_WORKSPACE_ID"] = userdata.get("LANGSMITH_WORKSPACE_ID")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
# @title Verify Environment Variables

import bcrypt
import os # Added os import

verify_hashed_keys = "True" # @param ["True", "False"]
should_verify_salts = verify_hashed_keys.lower() == "true"

for k, v in os.environ.items():
    if k.startswith("LANG") or k.startswith("OPEN") or k.startswith("TAVILY"):
        if k.endswith("KEY"):
            # Truncate the key to 72 bytes before hashing, as bcrypt has this limit.
            # Note: This is for demonstration of the bcrypt function and will not
            # produce a hash that can verify the full original key later.
            truncated_v = v.encode('utf-8')[:72]
            salted_key = bcrypt.hashpw(truncated_v, bcrypt.gensalt())
            if should_verify_salts:
                is_verified = bcrypt.checkpw(truncated_v, salted_key)
                print(f"{k}: {salted_key} - Verified: {is_verified}")
            else:
                print(f"{k}: {salted_key}")
        else:
            print(f"{k}: {v}")

LANGUAGE: en_US
LANG: en_US.UTF-8
LANGCHAIN_TRACING_V2: true
LANGCHAIN_TRACING: true
LANGSMITH_API_KEY: b'$2b$12$TNkDoPm.VPf8IWpwnRc.FedWmsaXgn3arPuFn2AVL80TwXg2kqqVq' - Verified: True
LANGSMITH_PROJECT: default
LANGSMITH_ENDPOINT: https://api.smith.langchain.com
LANGSMITH_WORKSPACE_ID: 5fbb32b2-4610-4bfb-8c3c-702249572f86
TAVILY_API_KEY: b'$2b$12$tgs/5Ppg4uSnIcH7wkIA6ecHEkpZbdck0/zHHIybdyHEuGQxUkLl6' - Verified: True
OPENAI_API_KEY: b'$2b$12$kcxZzUtr7Wxv8Y0DoUxiYuAiJgjjaMqwEL38wzktgF75oZyPxiRNm' - Verified: True


## 1) Hello LLM (baseline)

A single LLM call; no structure, no grounding.  
We'll improve over this baseline throughout the notebook.

In [ ]:

from langchain_openai import ChatOpenAI

# A deterministic model (temperature=0) for reproducible outputs.
llm = ChatOpenAI(temperature=0)

baseline_q = "What do you know about Langchain 1.0"
baseline_a = llm.invoke(baseline_q)
print("Q:", baseline_q)
print("\nBaseline (no context):\n", baseline_a.content)


Q: What do you know about Langchain 1.0

Baseline (no context):
 Langchain 1.0 is a blockchain platform that focuses on providing language services and solutions. It aims to create a decentralized ecosystem for language-related activities such as translation, interpretation, and language learning. The platform uses blockchain technology to ensure transparency, security, and efficiency in language services. Langchain 1.0 also offers a native cryptocurrency called LANG, which can be used for transactions within the platform.


## 2) Prompting + LCEL (LangChain Expression Language) + Output Parsing

Use **LCEL** (`|`) to pipe **PromptTemplate → LLM → OutputParser** so your code is composable and testable.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate   # ✅ v1 path
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# LLM (uses OPENAI_API_KEY from env)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Prompt → LLM → Parser, piped with LCEL `|`
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise technical assistant."),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

# Try it
chain.invoke({"question": "In one sentence, what does LangChain's Expression Language do? Show me an example"})


'LangChain\'s Expression Language allows users to create dynamic expressions for data manipulation and retrieval within language models. \n\n**Example:** `{{ user_input | to_uppercase | append: "!" }}` transforms the user input to uppercase and appends an exclamation mark.'

In [ ]:
prompt.invoke({"question": "What is today's date"})

ChatPromptValue(messages=[SystemMessage(content='You are a concise technical assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content="What is today's date", additional_kwargs={}, response_metadata={})])

In [ ]:
chain.invoke({"question": "What is LCEL in terms of Langchain"})

'In the context of Langchain, LCEL stands for "LangChain Event Log." It is a feature that allows developers to log events and interactions within a Langchain application. This can be useful for debugging, monitoring, and analyzing the behavior of the application, as it provides insights into how the components of the Langchain framework are interacting with each other and with external systems. LCEL can help in tracking the flow of data and understanding the performance of various components in a Langchain-based application.'

## 3) RAG

We’ll load a small corpus (LangSmith docs), split it, embed it, index it with **FAISS**, and wire a **Retrieval Chain**.

- This section runs **once** and is reused later by the Agent.
- If web loading fails, we fall back to a tiny local sample so the demo still runs.
- We'll also **surface sources** so you can see why answers improved.

In [ ]:
import os
os.environ.setdefault("USER_AGENT", "IK-LangChain-RAG/1.0 (contact: ops@your-org)")  # fixes the warning

'IK-LangChain-RAG/1.0 (contact: ops@your-org)'

In [ ]:
# Offline - Large data/documents - Do once
# RAG (v1): Web loader → splitter → FAISS → retriever → LCEL chain
from langchain_community.document_loaders import WebBaseLoader # Helps us download data from public pages and parsers for us easily
from langchain_text_splitters import RecursiveCharacterTextSplitter # Chunking Strategy
from langchain_openai import OpenAIEmbeddings, ChatOpenAI # Emb model - From OpenAI
from langchain_community.vectorstores import FAISS # VectorDB

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1) Load docs (pick any public pages you want indexed)
urls = [
    "https://python.langchain.com/docs/get_started/introduction/",
    "https://docs.smith.langchain.com/",
    "https://docs.langchain.com/oss/python/releases/langchain-v1",
    # "..."
]
loader = WebBaseLoader(urls, show_progress=True)
docs = loader.load()

# 2) Chunk
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200) # sliding window of 800 characters were we overlap by 200 characters
chunks = splitter.split_documents(docs)

# 3) Embed & index
emb = OpenAIEmbeddings()  # uses OPENAI_API_KEY from env
vs = FAISS.from_documents(chunks, emb)
# Vertex AI Search, Pinecone, Milvus

In [ ]:

# Online - At Runtime
retriever = vs.as_retriever(search_kwargs={"k": 4}) # returns top 4 relevant chunks to my query. Rc

In [ ]:
retriever.invoke("What is LangSmith?")

[Document(id='2f472fec-b64d-4d76-bee2-bd8f558060c7', metadata={'source': 'https://docs.smith.langchain.com/', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansAccount administrationOverviewSet up hierarchyWorkload isolationManage organizations using the APIManage billingGranular usageSet up resource tagsUser managementAdditional resourcesPolly AI assistantBetaData managementAccess control & AuthenticationScalability & resilienceFAQsRegions FAQPricing FAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for developing, debugging, and deploying AI agents and LLM applications.'),
 Document(id='bc72ac47-32f8-4818-

In [ ]:
# 4) Prompt (stuff-style: we inject all retrieved chunks into {context})
prompt_rag = ChatPromptTemplate.from_messages([
    ("system",
     "You are a precise assistant. Use the provided CONTEXT to answer.\n"
     "If the answer isn't in the context, say you don't know.\n\nCONTEXT:\n{context}"),
    ("human", "{question}")
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# 5) LCEL pipeline: {question} flows through; {context} is produced by retriever
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)

# 6) Try it
# rag_chain.invoke("What is LangSmith and how does it relate to LangChain?")
rag_chain.invoke("What do you know about Langchain 1.0") # {"question: '...'""}


"LangChain v1 is a focused, production-ready foundation for building agents. It features three core improvements and has streamlined the framework. There is a migration guide available for updating code to LangChain v1, and users are encouraged to report any issues discovered with version 1.0 on GitHub using the 'v1' label. Additional resources include documentation on middleware and agents."

In [ ]:
rag_chain.invoke("Who is the current president of the USA")

"I don't know."

## 4) Agent + Tools (on top of the same RAG)

We turn our retriever into a **Tool** and create an **OpenAI Functions Agent**.  
Optionally, if a **Tavily** API key is present, we add a web search tool.

> **Why an Agent?** It can decide *when* to use retrieval vs. answer directly, and sequence multi-step reasoning.

In [ ]:
import os, json
from typing import Any, Dict, List, Optional
from pprint import pprint

# LangChain components for building agents and tools
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool, create_retriever_tool
from langchain_core.messages import AIMessage, ToolMessage

# Agent
# - Work with tools
# - Find information from our VDB --> vdb_tool
# - Find it from online --> search_tool
# - Custom tools --> custom_tool
# - add all tools to a list

# Attempt to import TavilySearchResults for web search capabilities
# If Tavily is not installed or API key is missing, it will be skipped.
try:
    from langchain_community.tools.tavily_search import TavilySearchResults
except Exception:
    TavilySearchResults = None

# Initialize an empty list to store the tools the agent can use
tools: list[Any] = []

# Define a custom tool using the @tool decorator
@tool
def add(a: float, b: float) -> float:
    """Add two numbers a and b."""
    return a + b

@tool
def multiple(a: float, b: float) -> float:
    """Add two numbers a and b."""
    return a * b

@tool
def get_current_datetime():
    """ Always gets current datetime """
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

tools.append(get_current_datetime)

# Add the custom 'add' tool to the list of tools
tools.append(add)
tools.append(multiple)

# If a 'retriever' (from the RAG section) is available, create a knowledge base search tool
if "retriever" in globals():
    kb_tool = create_retriever_tool(
        globals()["retriever"], name="kb_search",
        description="Search the indexed KB/curriculum docs and return relevant passages."
    )
    tools.append(kb_tool)

# If TavilySearchResults is available and an API key is set, add a web search tool
if TavilySearchResults and os.getenv("TAVILY_API_KEY"):
    tools.append(TavilySearchResults(max_results=5, include_answer=True))

# Initialize the Language Model (LLM) for the agent
# Using gpt-4o-mini with temperature=0 for deterministic outputs.
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0)

# Create the agent with the LLM, defined tools, and a system prompt
# The system prompt guides the agent's behavior and tool usage strategy.
agent = create_agent(
    llm,
    tools,
    system_prompt=(
        "You are a helpful technical assistant. Use tools when useful. "
        "First get the current date time using the `get_current_datetime` tool. "
        "Prefer `kb_search` for questions about our course/KB."
        "Use the web search for general web info."
    ),
)


def _final_text(res: Any) -> str:
    """Extracts the final textual answer from an agent's response."""
    if isinstance(res, AIMessage):
        return res.content or ""
    if isinstance(res, dict) and "messages" in res:
        for m in reversed(res["messages"]):
            if isinstance(m, AIMessage) or getattr(m, "type", "") == "ai":
                return getattr(m, "content", "") or ""
    return str(res)

def _collect_tool_calls_and_outputs(res: Any, max_len: int = 500) -> List[Dict[str, Any]]:
    """Collects all tool calls and their corresponding outputs from an agent's response."""
    messages: List[Any] = []
    if isinstance(res, dict) and "messages" in res:
        messages = res["messages"]

    tool_calls: List[Dict[str, Any]] = []
    # Find the AI message that contains tool calls
    for m in messages:
        if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
            tool_calls = m.tool_calls or []
            break
        # Handle older versions or different message formats that might store tool_calls in additional_kwargs
        addkw = getattr(m, "additional_kwargs", {}) if hasattr(m, "additional_kwargs") else {}
        if addkw.get("tool_calls"):
            tool_calls = addkw["tool_calls"]
            break

    # Map tool call IDs to their outputs from ToolMessage objects
    outputs: Dict[str, str] = {}
    for m in messages:
        if isinstance(m, ToolMessage):
            outputs[getattr(m, "tool_call_id", None)] = (getattr(m, "content", "") or "")

    def trunc(s: Optional[str]) -> str:
        """Truncates a string to a maximum length with an ellipsis."""
        if not s:
            return ""
        return s[:max_len] + ("…" if len(s) > max_len else "")

    # Structure the collected tool call information with their outputs
    structured: List[Dict[str, Any]] = []
    for c in tool_calls:
        structured.append({
            "name": c.get("name"), "args": c.get("args"),
            "output": trunc(outputs.get(c.get("id")))
        })
    return structured

def _usage(res: Any) -> Optional[Dict[str, Any]]:
    """Extracts token usage information from an agent's response."""
    if isinstance(res, AIMessage):
        return getattr(res, "response_metadata", {}).get("token_usage")
    if isinstance(res, dict) and "messages" in res:
        # Iterate messages in reverse to find the last message with usage data
        for m in reversed(res["messages"]):
            meta = getattr(m, "response_metadata", {}) if hasattr(m, "response_metadata") else {}
            if meta.get("token_usage"):
                return meta["token_usage"]
    return None

def to_structured(res: Any) -> Dict[str, Any]:
    """
    Formats the raw agent response into a structured dictionary for easier analysis.

    Args:
        res: The raw response object from the agent, which can be an AIMessage or a dictionary.

    Returns:
        A dictionary containing the final answer, a list of tool calls with their outputs, and token usage statistics.
    """
    return {"answer": _final_text(res), "tools": _collect_tool_calls_and_outputs(res), "usage": _usage(res)}

/tmp/ipython-input-3243970127.py:61: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tools.append(TavilySearchResults(max_results=5, include_answer=True))


In [ ]:
# Example query to demonstrate agent's capabilities
query = "What is the current time. What is 41 + 1? Also, if I ask about LangGraph later, how would you use kb_search?"

# Invoke the agent with the query
res = agent.invoke({"messages": [{"role": "user", "content": query}]})

# Print the structured result for easy inspection
pprint(to_structured(res), width=100)

{'answer': 'The current time is **2026-02-22 20:58:19**. The result of \\(41 + 1\\) is **42**.\n'
           '\n'
           'If you ask about LangGraph later, I would use the `kb_search` tool to look up relevant '
           'information in our course or knowledge base. This would help me provide you with '
           'accurate and specific details related to LangGraph.',
 'tools': [{'args': {}, 'name': 'get_current_datetime', 'output': '2026-02-22 20:58:19'},
           {'args': {'a': 41, 'b': 1}, 'name': 'add', 'output': '42.0'}],
 'usage': {'completion_tokens': 80,
           'completion_tokens_details': {'accepted_prediction_tokens': 0,
                                         'audio_tokens': 0,
                                         'reasoning_tokens': 0,
                                         'rejected_prediction_tokens': 0},
           'prompt_tokens': 330,
           'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
           'total_tokens': 410}}


In [ ]:
from pprint import pprint
from langchain_core.messages import AIMessage

queries = [
    "List two LangSmith capabilities that support evaluation and how to use them.",
    "Where do the docs explain tracing? Summarize in 3 bullets.",
]

def _final_text(res):
    if isinstance(res, AIMessage):
        return res.content or ""
    if isinstance(res, dict) and "messages" in res:
        for m in reversed(res["messages"]):
            if isinstance(m, AIMessage) or getattr(m, "type", "") == "ai":
                return getattr(m, "content", "") or ""
    return str(res)

for q in queries:
    print("\n" + "=" * 26)
    print("AGENT Q:", q)

    res = agent.invoke({"messages": [{"role": "user", "content": q}]})

    if "to_structured" in globals():
        pprint(to_structured(res), width=100)
    else:
        print("\nAGENT A:\n", _final_text(res))


AGENT Q: List two LangSmith capabilities that support evaluation and how to use them.
{'answer': 'LangSmith offers several capabilities that support evaluation in AI development. Here '
           'are two key capabilities along with how to use them:\n'
           '\n'
           '1. **Prompt Testing**:\n'
           '   - **Description**: This feature allows users to iterate on prompts with built-in '
           'versioning and collaboration tools. It helps in refining prompts to improve the '
           'performance of AI models.\n'
           '   - **How to Use**: You can access the prompt testing feature within the LangSmith '
           'Studio. Here, you can create different versions of your prompts, test them against '
           'various inputs, and analyze the outputs. This iterative process enables you to make '
           'data-driven improvements to your prompts quickly.\n'
           '\n'
           '2. **Integrated Monitoring and Evaluation**:\n'
           '   - **Descr

## 5) Tiny UI (Gradio)

A minimal chat interface that routes user messages to the agent.  
If Tavily is not available, the agent still works with the retriever tool.

In [ ]:
import gradio as gr
from langchain_core.messages import AIMessage

def _final_text(res):
    if isinstance(res, AIMessage):
        return res.content or ""
    if isinstance(res, dict) and "messages" in res:
        for m in reversed(res["messages"]):
            if isinstance(m, AIMessage) or getattr(m, "type", "") == "ai":
                return getattr(m, "content", "") or ""
    return str(res)

def _to_messages(history, message):
    """
    Converts the chat history and user message into a list of messages for the agent.

    Args:
        history: The chat history as a list of (user_message, agent_message) tuples.
        message: The user's message as a string.

    Returns:
        A list of messages in the format expected by the agent.
        Each message is a dictionary with keys:
        role: The role of the message sender ("user" or "assistant").
        content: The content of the message.

    Example:
        history = [
            ("User 1", "Agent 1"),
            ("User 2", "Agent 2"),
        ]


      user: The user's message.
      assistant: The agent's response.


    """
    msgs = []
    for u, a in history:
        if u: msgs.append({"role": "user", "content": u})
        if a: msgs.append({"role": "assistant", "content": a})
    msgs.append({"role": "user", "content": message})
    return msgs

def _ensure_agent():
    global agent
    try:
        # Attempt to return the globally defined agent if it exists
        agent
        return agent
    except NameError:
        # If 'agent' is not defined, create a new one with available tools
        from langchain_openai import ChatOpenAI
        from langchain.agents import create_agent
        from langchain_core.tools import tool, create_retriever_tool
        import os

        # Initialize tools list for the fallback agent
        fallback_tools = []

        @tool
        def get_current_datetime():
          """ Always gets current datetime """
          from datetime import datetime
          return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        fallback_tools.append(add)

        @tool
        def add(a: float, b: float) -> float:
            "Add two numbers." # Docstring added
            return a + b
        fallback_tools.append(add)

        @tool
        def multiply(a: float, b: float) -> float:
            "Multiplies two numbers." # Docstring added
            return a * b
        fallback_tools.append(multiply)

        # Conditionally add kb_search tool if retriever is available
        if "retriever" in globals():
            fallback_tools.append(create_retriever_tool(
                globals()["retriever"], name="kb_search",
                description="Search the indexed KB/curriculum docs and return relevant passages."
            ))

        # Conditionally add TavilySearchResults for web search
        try:
            from langchain_community.tools.tavily_search import TavilySearchResults
            if os.getenv("TAVILY_API_KEY"):
                fallback_tools.append(TavilySearchResults(max_results=5, include_answer=True))
        except Exception:
            pass # Tavily not available or API key not set

        # Initialize LLM for the fallback agent
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

        # Create the fallback agent
        agent = create_agent(llm, fallback_tools, system_prompt=("You are a helpful technical assistant. Use tools when useful. "
                                                                "Make sure you call get_current_datetime tool to get the current time before answering questions. "
                                                               "Prefer kb_search for questions about our course/KB; use web search for general web info."))
        return agent

def chat_fn(message, history):
    try:
        ag = _ensure_agent()
        msgs = _to_messages(history, message)
        res = ag.invoke({"messages": msgs})
        return _final_text(res)
    except Exception as e:
        return f"Error: {e}"

# Check if demo was previously defined in the global scope before trying to close it.
if 'demo' in globals() and isinstance(demo, gr.Blocks):
    try:
        demo.close()
    except Exception:
        pass

with gr.Blocks() as demo:
    gr.Markdown("# LangChain Agent Chat")
    gr.Markdown("Ask about your KB (kb_search) or general queries. Web search only if TAVILY_API_KEY is set.")
    gr.ChatInterface(chat_fn)
    gr.Markdown('Tip: Try "Where are tracing docs?" or "Add 3.5 and 4."')

demo.launch(share=True)

## 6) Custom Tool (@tool) example

One simple tool is enough to demonstrate schema and descriptions.  
The agent can call this tool if it detects a matching need.

## 7) Custom tools

Define new tools with the `@tool` decorator. Rebuild an agent by passing the updated tools list to `create_agent`, then invoke.


In [ ]:
import os, json
from typing import Any, Dict, List, Optional

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent  # Changed import
from langchain_core.tools import tool, create_retriever_tool
from langchain_core.messages import AIMessage, ToolMessage

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers a and b."""
    return a * b

tools2: List[Any] = []
if "tools" in globals():
    tools2.extend(globals()["tools"])
tools2.append(multiply)

if "retriever" in globals() and not any(getattr(t, "name", "") == "kb_search" for t in tools2):
    tools2.append(create_retriever_tool(
        retriever,
        name="kb_search",
        description="Search the indexed KB/curriculum docs and return relevant passages."
    ))

try:
    from langchain_community.tools.tavily_search import TavilySearchResults
    if os.getenv("TAVILY_API_KEY") and not any(getattr(t, "name", "") == "tavily_search_results_json" for t in tools2):
        tools2.append(TavilySearchResults(max_results=5, include_answer=True))
except Exception:
    pass

try:
    llm
except NameError:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

SYSTEM_PROMPT = (
    "You are a helpful technical assistant. Use tools when useful. "
    "Prefer kb_search for questions about our course/KB; use web search for general web info."
)
agent2 = create_agent(llm, tools2, system_prompt=SYSTEM_PROMPT)  # Changed function and parameter

def _final_text(res: Any) -> str:
    if isinstance(res, AIMessage):
        return res.content or ""
    if isinstance(res, dict) and "messages" in res:
        for m in reversed(res["messages"]):
            if isinstance(m, AIMessage) or getattr(m, "type", "") == "ai":
                return getattr(m, "content", "") or ""
    return str(res)

def _collect_tool_calls_and_outputs(res: Any, max_len: int = 500) -> List[Dict[str, Any]]:
    messages: List[Any] = res.get("messages", []) if isinstance(res, dict) else []

    tool_calls = []
    for m in messages:
        if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
            tool_calls = m.tool_calls or []
            break
        ak = getattr(m, "additional_kwargs", {}) if hasattr(m, "additional_kwargs") else {}
        if ak.get("tool_calls"):
            tool_calls = ak["tool_calls"]; break

    outputs = {getattr(m, "tool_call_id", None): (getattr(m, "content", "") or "")
               for m in messages if isinstance(m, ToolMessage)}

    def trunc(s: Optional[str]) -> str:
        if not s: return ""
        return s[:max_len] + ("…" if len(s) > max_len else "")

    return [{"name": c.get("name"), "args": c.get("args"), "output": trunc(outputs.get(c.get("id")))}
            for c in tool_calls]

def _usage(res: Any) -> Optional[Dict[str, Any]]:
    if isinstance(res, AIMessage):
        return getattr(res, "response_metadata", {}).get("token_usage")
    if isinstance(res, dict) and "messages" in res:
        for m in reversed(res["messages"]):
            meta = getattr(m, "response_metadata", {}) if hasattr(m, "response_metadata") else {}
            if meta.get("token_usage"):
                return meta["token_usage"]
    return None

def to_structured(res: Any) -> Dict[str, Any]:
    return {"answer": _final_text(res), "tools": _collect_tool_calls_and_outputs(res), "usage": _usage(res)}

q = "Multiply 3.5 by 4 and then list two LangSmith evaluation features."
res = agent2.invoke({"messages": [{"role": "user", "content": q}]})
print(json.dumps(to_structured(res), indent=2))

{
  "answer": "The result of multiplying 3.5 by 4 is **14.0**.\n\nAs for LangSmith evaluation features, here are two notable ones:\n\n1. **Prompt Testing**: LangSmith allows users to iterate on prompts with built-in versioning and collaboration, enabling faster improvements and testing of prompts.\n\n2. **Integrated Monitoring and Evaluation**: It provides tools for tracing requests, evaluating outputs, and managing deployments all in one place, which helps in building more reliable AI systems.",
  "tools": [
    {
      "name": "multiply",
      "args": {
        "a": 3.5,
        "b": 4
      },
      "output": "14.0"
    },
    {
      "name": "kb_search",
      "args": {
        "query": "LangSmith evaluation features"
      },
      "output": "LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...\u2318KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringDeploymentPl

## 7) Wrap-up & Next steps

You've successfully built an end-to-end AI application in this notebook:
-   Progressed from a baseline LLM call to a **prompted chain**, then to **Retrieval-Augmented Generation (RAG)**, and finally to an **Agent with tools**.
-   Integrated an **(Optional) UI** using Gradio for interactive chat.
-   **Re-used the same retriever** efficiently across different components.
-   **Optionally enabled LangSmith** tracing for enhanced observability.

### Ideas to extend this project:
-   **Swap out FAISS** for your preferred vector database.
-   Incorporate **validators** (output schemas) and develop **evaluation** suites to rigorously test your application.
-   Add more **domain-specific tools**, such as database query tools, advanced calculators, or integrations with internal APIs, to expand your agent's capabilities.